# 🎓 1st Project — **2nd Delivery**


### Teacher
- Cristian Camilo Zapata Zuluaga

### Students
- Roi Jared Flores Garza Stone
- Ivan Morales

November 5th, 2025, ITESO

# Models Training

Working with Mlflow through DataBricks

In [26]:
import os, mlflow
from dotenv import load_dotenv

load_dotenv(override=True) # Cargar las variables de entorno desde el archivo .env
EXPERIMENT_NAME = "/Users/ivan.morales@iteso.mx/coffee-intake-experiments" 

mlflow.set_tracking_uri("databricks")
experiment = mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

## Import required packages

In [27]:
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import pandas as pd
import numpy as np
import sys
from functools import partial
from hyperopt import fmin, tpe, STATUS_OK, hp, Trials
from hyperopt.pyll.base import scope 
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import f1_score

We wil import the objects used in ```_02_data_wrangling``` to create the full pipeline for the model training

In [28]:
# Obtener la ruta del directorio padre (un nivel 'arriba' de tu notebook)
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Añadir el directorio padre al 'path' de Python
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
    
from preprocessing import ct as preprocessor

In [29]:
data = pd.read_csv("../data/processed/processed_coffee_data.csv")

X = data.drop("ordinal_encoding__Sleep_Quality", axis=1)
y = data[["ordinal_encoding__Sleep_Quality"]]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=33)

In [30]:
from functools import partial
from hyperopt import fmin, tpe, STATUS_OK, hp, Trials
from hyperopt.pyll.base import scope 

In [31]:
space_logreg = {
    'C': hp.loguniform('logreg_C', np.log(0.01), np.log(100)),
    'penalty': hp.choice('logreg_penalty', ['l1', 'l2']),
    'solver': 'liblinear',
    'max_iter': 1000
}

space_rf = {
    'n_estimators': scope.int(hp.quniform('rf_n_estimators', 50, 500, 25)),
    'max_depth': scope.int(hp.quniform('rf_max_depth', 3, 30, 1)),
    'max_features': hp.choice('rf_max_features', ['sqrt', 'log2', None]),
    'criterion': hp.choice('rf_criterion', ['gini', 'entropy'])
}

space_mlp = {
    'hidden_layer_sizes': hp.choice('mlp_hidden_layers', [
        (50,),               # 1 layer of 50 neurons
        (100,),              # 1 layer of 100 neurons
        (50, 50),            # 2 layers of 50 neurons
        (100, 50),           # 2 layers of 100 and 50
        (100, 100, 50)       # 3 layers
    ]),
    'activation': hp.choice('mlp_activation', ['relu', 'tanh']),
    'alpha': hp.loguniform('mlp_alpha', np.log(0.0001), np.log(0.1)), # L2 regularization
    'learning_rate_init': hp.loguniform('mlp_lr_init', np.log(0.001), np.log(0.1)),
    
    # --- Fixed parameters for stability and speed ---
    'solver': 'adam',
    'max_iter': 500,           # Give it enough time to converge
    'early_stopping': True,    # CRITICAL: stops bad trials early
    'validation_fraction': 0.1 # Used for early stopping
}

In [32]:
model_config = {
    'LogisticRegression': {
        'model_class': LogisticRegression,
        'search_space': space_logreg
    },
    'RandomForest': {
        'model_class': RandomForestClassifier,
        'search_space': space_rf
    },
    'MLP': {
        'model_class': MLPClassifier,
        'search_space': space_mlp
    }
}

In [33]:
def objective(params, model_class, model_name):
    
    # --- This is the key to MLflow nested runs ---
    with mlflow.start_run(run_name=f"{model_name}_Trial", nested=True):
        
        # Hyperopt gives 'raw' params. We must clean them for sklearn.
        params_cleaned = params.copy()
                    
        # Log the cleaned params
        mlflow.log_params(params_cleaned)

        # --- 4b. Train and Evaluate Model ---
        model = model_class(**params_cleaned, random_state=42)
        
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        metric = f1_score(y_val, preds, average="weighted")
        
        # Log the metric
        mlflow.log_metric("accuracy", metric)

        # --- 4c. Return Loss for Hyperopt ---
        # Hyperopt *minimizes*, so we return the *negative* of our metric
        loss = -metric
        
        return {'loss': loss, 'status': STATUS_OK}

In [ ]:
MAX_EVALS_PER_MODEL = 25 # Set how many trials per model

for model_name, config in model_config.items():
    
    # --- 5a. Start the PARENT Run ---
    with mlflow.start_run(run_name=f"{model_name}_HPO") as parent_run:
        
        print(f"\n--- 🏃 Starting Parent Run for: {model_name} ---")

        # Create a Trials object to store fmin history
        trials = Trials()
        
        # This "freezes" the model_class and model_name arguments
        # so that 'fmin' can call objective(params) correctly.
        fmin_objective = partial(objective, 
                                 model_class=config['model_class'], 
                                 model_name=model_name)

        # This will automatically create all the nested CHILD runs
        best_params = fmin(
            fn=fmin_objective,
            space=config['search_space'],
            algo=tpe.suggest,
            max_evals=MAX_EVALS_PER_MODEL,
            trials=trials
        )
        
        # --- 5d. Log Best Results to PARENT Run ---
        print(f"--- 🏆 Best results for {model_name} ---")
        
        # Get the best trial's loss (accuracy)
        best_loss = trials.best_trial['result']['loss']
        best_accuracy = -best_loss
        print(f"Best F1_core: {best_accuracy:.4f}")
        
        mlflow.log_metric("F1_score", best_accuracy)
        
        # Clean the 'best' params (which are in 'hyperopt' format)
        best_params_cleaned = best_params.copy()            
        mlflow.log_params(best_params_cleaned)

print("\n--- ✅ Experiment Finished ---")


--- 🏃 Starting Parent Run for: LogisticRegression ---
  0%|          | 0/25 [00:00<?, ?trial/s, best loss=?]

c:\Users\Roi_f\PCD\coffee_intake_project\.venv\Lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)

c:\Users\Roi_f\PCD\coffee_intake_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(



🏃 View run LogisticRegression_Trial at: https://dbc-ed7122c4-c5b7.cloud.databricks.com/ml/experiments/1704736485671243/runs/136d8fe6319243b682b25395420d35b6

🧪 View experiment at: https://dbc-ed7122c4-c5b7.cloud.databricks.com/ml/experiments/1704736485671243

  4%|▍         | 1/25 [00:01<00:24,  1.00s/trial, best loss: -0.9751987067331044]

c:\Users\Roi_f\PCD\coffee_intake_project\.venv\Lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)

c:\Users\Roi_f\PCD\coffee_intake_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(



🏃 View run LogisticRegression_Trial at: https://dbc-ed7122c4-c5b7.cloud.databricks.com/ml/experiments/1704736485671243/runs/04c722ee75904c55a2c9ff31ee26e76e

🧪 View experiment at: https://dbc-ed7122c4-c5b7.cloud.databricks.com/ml/experiments/1704736485671243

  8%|▊         | 2/25 [00:01<00:22,  1.03trial/s, best loss: -0.982712294968761] 

...

c:\Users\Roi_f\PCD\coffee_intake_project\.venv\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)



🏃 View run RandomForest_Trial at: https://dbc-ed7122c4-c5b7.cloud.databricks.com/ml/experiments/1704736485671243/runs/21de9147f6f04b66981c76951c5cac6a

🧪 View experiment at: https://dbc-ed7122c4-c5b7.cloud.databricks.com/ml/experiments/1704736485671243

 64%|██████▍   | 16/25 [01:10<00:41,  4.63s/trial, best loss: -0.995973241738855]

c:\Users\Roi_f\PCD\coffee_intake_project\.venv\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)



🏃 View run RandomForest_Trial at: https://dbc-ed7122c4-c5b7.cloud.databricks.com/ml/experiments/1704736485671243/runs/3ce42acd2b934e369499e8b048ed2455

🧪 View experiment at: https://dbc-ed7122c4-c5b7.cloud.databricks.com/ml/experiments/1704736485671243

 68%|██████▊   | 17/25 [01:17<00:43,  5.41s/trial, best loss: -0.995973241738855]

c:\Users\Roi_f\PCD\coffee_intake_project\.venv\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)



In [ ]:
model_registry = "workspace.default.coffee/sleep_quality-model"
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by= ["metrics.F1_score DESC"],
    output_format="list"
)

if runs > 0:
    best_run = runs[0]
    second_best = runs[1]

In [ ]:
result_champ = mlflow.register_model(
    model_uri=f"runs:/{best_run.info.run_id}/model",
    name=model_registry
)

result_chall = mlflow.register_model(
    model_uri=f"runs:/{second_best.info.run_id}/model",
    name=model_registry
)

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()
model_chall_version = result_chall.version
model_champ_version = result_champ.version
challenger_alias ="Challenger"
champ_alias ="Champion"


client.set_registered_model_alias(
    name=model_registry,
    alias=challenger_alias,
    version=model_chall_version
)

client.set_registered_model_alias(
    name=model_registry,
    alias=champ_alias,
    version= model_champ_version
)